# Token Pruning — DynamicViT on CheXpert

Rao et al. (2021) DynamicViT. Trains three models (keep ratios 70 / 50 / 30 %)
each starting from the baseline checkpoint. Two lightweight prediction MLPs
(after blocks 4 and 8) learn which patch tokens to drop.

**Runtime on L4 GPU**: ~3 hours per ratio → ~9 hours total. Run overnight.
All checkpoints and masks save to Google Drive as they complete.

**Outputs**
- `checkpoints/dynvit_{ratio}pct_best.pt` — trained model per ratio
- `maps/token_masks/masks_{ratio}pct.npy` — [202, 14, 14] binary masks (val set)
- `maps/gradcam/` — Grad-CAM maps from each pruned model
- `checkpoints/token_pruning_iou.csv` — per-label IoU vs baseline Grad-CAM
- `checkpoints/token_pruning_auc.csv` — per-label AUC comparison

In [ ]:
# ── Cell 1 — Environment ──────────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/dizertatie_project'):
        !git clone https://github.com/daviDpaD18/diz.git /content/diz
        os.symlink('/content/diz/dizertatie_project', '/content/dizertatie_project')
    sys.path.insert(0, '/content/dizertatie_project')
except ImportError:
    IS_COLAB = False
    sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))

print('Colab:', IS_COLAB)

In [ ]:
# ── Cell 2 — Data to local SSD ────────────────────────────────────────────────
if IS_COLAB:
    DRIVE = '/content/drive/MyDrive/dizertatie'
    if not os.path.exists('/content/train'):
        print('Copying train.zip ...')
        !cp {DRIVE}/train.zip /content/train.zip
        !unzip -q /content/train.zip -d /content/
        !rm /content/train.zip
        print('train/ ready.')
    if not os.path.exists('/content/valid'):
        !cp -r {DRIVE}/valid /content/valid
        print('valid/ ready.')
    import importlib, src.config
    importlib.reload(src.config)

!pip install -q grad-cam

import json, numpy as np, pandas as pd, warnings
import torch, torch.nn.functional as F, timm
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import roc_auc_score
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

from src.config import IMAGE_ROOT, SPLITS_DIR, WEIGHTS_DIR, CKPT_DIR
from src.dataset import LABEL_COLS, CheXpertDataset, train_transforms, eval_transforms
from src.dynamic_vit import DynamicViT, ratio_loss

CKPT_DIR.mkdir(parents=True, exist_ok=True)
MAPS_DIR = (CKPT_DIR.parent / 'maps') if IS_COLAB \
           else Path('..').resolve() / 'maps'
(MAPS_DIR / 'token_masks').mkdir(parents=True, exist_ok=True)
(MAPS_DIR / 'gradcam').mkdir(parents=True, exist_ok=True)

DEVICE = (
    torch.device('cuda')  if torch.cuda.is_available()  else
    torch.device('mps')   if torch.backends.mps.is_available() else
    torch.device('cpu')
)
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_available() else None
print(f'Device: {DEVICE}  AMP: {AMP_DTYPE}')

In [ ]:
# ── Cell 3 — Constants ────────────────────────────────────────────────────────
KEEP_RATIOS    = [0.70, 0.50, 0.30]   # fraction of 196 tokens to keep
TRAIN_EPOCHS   = 10 if IS_COLAB else 1
BATCH_SIZE     = 32 if IS_COLAB else 8
NUM_WORKERS    = 4  if IS_COLAB else 0
LR             = 1e-4
WEIGHT_DECAY   = 1e-2
GRAD_CLIP      = 1.0
RATIO_LAM      = 2.0    # weight of keeping-ratio loss (DynamicViT paper default)

# Best baseline checkpoint — all token-pruning models start from here
seeds = [42, 123, 456]
best_seed, best_auc = max(
    ((s, torch.load(CKPT_DIR / f'baseline_seed{s}_best.pt',
                    map_location='cpu', weights_only=False)['val_macro_auc'])
     for s in seeds),
    key=lambda x: x[1]
)
BASELINE_CKPT = CKPT_DIR / f'baseline_seed{best_seed}_best.pt'
print(f'Baseline: seed {best_seed}, AUC {best_auc:.4f}')
print(f'Keep ratios: {[int(r*100) for r in KEEP_RATIOS]}%')
print(f'Tokens kept: {[int(r*196) for r in KEEP_RATIOS]} of 196')

# Class weights
with open(WEIGHTS_DIR / 'class_weights.json') as f:
    cw = json.load(f)
CLASS_WEIGHTS = torch.tensor([cw[l] for l in LABEL_COLS], dtype=torch.float32).to(DEVICE)

def weighted_bce(logits, labels):
    return (F.binary_cross_entropy_with_logits(
        logits, labels, reduction='none') * CLASS_WEIGHTS).mean()

In [ ]:
# ── Cell 4 — Data loaders ─────────────────────────────────────────────────────
TRAIN_CSV = SPLITS_DIR / ('train_full.csv'   if IS_COLAB else 'train_dev5k.csv')
VALID_CSV = SPLITS_DIR / 'valid_frontal.csv'

df_train = pd.read_csv(TRAIN_CSV)
df_valid = pd.read_csv(VALID_CSV)

train_ds     = CheXpertDataset(df_train, IMAGE_ROOT, train_transforms)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=IS_COLAB, drop_last=True)
val_ds       = CheXpertDataset(df_valid, IMAGE_ROOT, eval_transforms)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=IS_COLAB)
val_loader_1 = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

print(f'Train: {len(df_train):,}  Val: {len(df_valid)}')

In [ ]:
# ── Cell 5 — Validation helper ────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels, *_ in val_loader:
        logits = model(imgs.to(DEVICE))
        all_probs.append(torch.sigmoid(logits).cpu())
        all_labels.append(labels)
    probs  = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    auc = {}
    for i, label in enumerate(LABEL_COLS):
        auc[label] = roc_auc_score(labels[:, i], probs[:, i]) \
                     if labels[:, i].sum() > 0 else float('nan')
    return float(np.nanmean(list(auc.values()))), auc

In [ ]:
# ── Cell 6 — Training loop (all three keep ratios) ────────────────────────────
#
# Each ratio trains independently from the baseline checkpoint.
# Skips a ratio if its best checkpoint already exists on Drive.
# Loss = weighted_BCE + ratio_loss (keeps the model honest about
# how many tokens it actually drops).

all_train_results = {}

for keep_ratio in KEEP_RATIOS:
    pct       = int(keep_ratio * 100)
    save_path = CKPT_DIR / f'dynvit_{pct}pct_best.pt'

    if save_path.exists():
        ckpt = torch.load(save_path, map_location='cpu', weights_only=False)
        print(f'[{pct}%] Already trained — AUC {ckpt["val_macro_auc"]:.4f}. Skipping.')
        all_train_results[pct] = ckpt['val_macro_auc']
        continue

    print(f'\n{"="*60}\nTOKEN PRUNING {pct}%  ({int(keep_ratio*196)} / 196 tokens kept)\n{"="*60}')

    # Build DynamicViT from baseline weights
    base_ckpt  = torch.load(BASELINE_CKPT, map_location='cpu', weights_only=False)
    base_model = timm.create_model(base_ckpt['hparams']['model'],
                                   pretrained=False, num_classes=14)
    base_model.load_state_dict(base_ckpt['model_state_dict'])
    model = DynamicViT(base_model, final_keep_ratio=keep_ratio).to(DEVICE)

    # Only the prediction MLPs are new — give backbone a lower LR
    predictor_params = list(model.predictors.parameters())
    predictor_ids    = {id(p) for p in predictor_params}
    backbone_params  = [p for p in model.parameters() if id(p) not in predictor_ids]

    optimizer = torch.optim.AdamW([
        {'params': backbone_params,  'lr': LR * 0.1},   # pretrained — lower LR
        {'params': predictor_params, 'lr': LR},          # new predictors — full LR
    ], weight_decay=WEIGHT_DECAY)

    total_steps = TRAIN_EPOCHS * len(train_loader)
    scheduler   = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.05 * total_steps),
        num_training_steps=total_steps)
    scaler = torch.amp.GradScaler() if torch.cuda.is_available() else None

    best_auc_dv = 0.0

    for epoch in range(1, TRAIN_EPOCHS + 1):
        model.train()
        running_cls = running_rat = 0.0

        for imgs, labels, *_ in tqdm(train_loader,
                                      desc=f'[{pct}%] Epoch {epoch}/{TRAIN_EPOCHS}',
                                      leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()

            if scaler:
                with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                    logits, decisions = model(imgs, return_decisions=True)
                    loss_cls = weighted_bce(logits, labels)
                    loss_rat = ratio_loss(decisions, keep_ratio, RATIO_LAM)
                    loss     = loss_cls + loss_rat
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer); scaler.update()
            else:
                logits, decisions = model(imgs, return_decisions=True)
                loss_cls = weighted_bce(logits, labels)
                loss_rat = ratio_loss(decisions, keep_ratio, RATIO_LAM)
                loss     = loss_cls + loss_rat
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

            scheduler.step()
            running_cls += loss_cls.item()
            running_rat += loss_rat.item()

        macro, auc_per_label = evaluate(model)
        n = len(train_loader)
        flag = ' ← best' if macro > best_auc_dv else ''
        print(f'  Epoch {epoch:>2} | cls {running_cls/n:.4f} '
              f'| ratio {running_rat/n:.4f} | AUC {macro:.4f}{flag}')

        if macro > best_auc_dv:
            best_auc_dv = macro
            torch.save({
                'keep_ratio':        keep_ratio,
                'epoch':             epoch,
                'model_state_dict':  model.state_dict(),
                'val_macro_auc':     macro,
                'val_auc_per_label': auc_per_label,
                'hparams':           base_ckpt['hparams'],
            }, save_path)

    all_train_results[pct] = best_auc_dv
    print(f'[{pct}%] Done. Best AUC: {best_auc_dv:.4f}')

In [ ]:
# ── Cell 7 — AUC comparison table ────────────────────────────────────────────
RELIABLE_MIN   = 5
val_pos        = df_valid[LABEL_COLS].sum().to_dict()
ckpt_names_dv  = ['baseline'] + [f'dynvit_{int(r*100)}pct' for r in KEEP_RATIOS]
ckpt_paths_dv  = [BASELINE_CKPT] + \
                 [CKPT_DIR / f'dynvit_{int(r*100)}pct_best.pt' for r in KEEP_RATIOS]

auc_table = {}
for name, path in zip(ckpt_names_dv, ckpt_paths_dv):
    d = torch.load(path, map_location='cpu', weights_only=False)
    for label, val in d['val_auc_per_label'].items():
        auc_table.setdefault(label, {})[name] = val

rows = []
for label in LABEL_COLS:
    n_pos = int(val_pos.get(label, 0))
    row   = {'Label': label, 'Val+': n_pos,
             'Reliable': '✓' if n_pos >= RELIABLE_MIN else f'✗ (n={n_pos})'}
    for name in ckpt_names_dv:
        row[name] = auc_table.get(label, {}).get(name, float('nan'))
    base = row['baseline']
    for name in ckpt_names_dv[1:]:
        row[f'Δ{name}'] = row[name] - base if not np.isnan(row[name]) else float('nan')
    rows.append(row)

df_auc = pd.DataFrame(rows).set_index('Label')
show_cols = ['Val+', 'Reliable', 'baseline'] + \
            [f'dynvit_{int(r*100)}pct' for r in KEEP_RATIOS] + \
            [f'Δdynvit_{int(r*100)}pct' for r in KEEP_RATIOS]

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print(df_auc[show_cols].round(4).to_string())

for name in ckpt_names_dv:
    vals = [auc_table.get(l, {}).get(name, float('nan'))
            for l in LABEL_COLS if int(val_pos.get(l, 0)) >= RELIABLE_MIN]
    with warnings.catch_warnings(): warnings.simplefilter('ignore')
    print(f'Macro AUC (reliable, {name}): {np.nanmean(vals):.4f}')

df_auc[show_cols].to_csv(CKPT_DIR / 'token_pruning_auc.csv')
print('Saved token_pruning_auc.csv')

In [ ]:
# ── Cell 8 — Extract and save token masks ─────────────────────────────────────
#
# For each pruned checkpoint, run inference on all 202 val images and
# record which patch tokens survived. Saved as [202, 14, 14] binary arrays.
# These are used in Cell 10 for the IoU comparison against Grad-CAM.

token_masks = {}

for keep_ratio in KEEP_RATIOS:
    pct  = int(keep_ratio * 100)
    path = CKPT_DIR / f'dynvit_{pct}pct_best.pt'

    ckpt_d     = torch.load(path, map_location='cpu', weights_only=False)
    base_model = timm.create_model(ckpt_d['hparams']['model'],
                                   pretrained=False, num_classes=14)
    model_dv   = DynamicViT(base_model, final_keep_ratio=keep_ratio).to(DEVICE)
    model_dv.load_state_dict(ckpt_d['model_state_dict'])
    model_dv.eval()

    masks = []
    for imgs, *_ in tqdm(val_loader_1, desc=f'Token masks [{pct}%]'):
        mask = model_dv.get_token_mask(imgs.to(DEVICE))   # [1, 14, 14]
        masks.append(mask.cpu().numpy())

    arr = np.concatenate(masks, axis=0)   # [202, 14, 14]
    token_masks[pct] = arr
    save_p = MAPS_DIR / 'token_masks' / f'masks_{pct}pct.npy'
    np.save(save_p, arr)

    n_kept = arr.sum(axis=(1, 2)).mean()
    print(f'[{pct}%] Saved {arr.shape}  avg tokens kept: {n_kept:.1f} / 196  '
          f'(target {int(keep_ratio*196)})')

In [ ]:
# ── Cell 9 — Load baseline Grad-CAM maps ─────────────────────────────────────
#
# The baseline Grad-CAM maps were generated in notebook 03 and saved on Drive.
# If they don't exist yet, we regenerate them here.

baseline_gcam_path = MAPS_DIR / 'gradcam' / 'gradcam_baseline.npy'

if baseline_gcam_path.exists():
    baseline_gcam = np.load(baseline_gcam_path)   # [202, 14, 14, 14]
    print(f'Loaded baseline Grad-CAM: {baseline_gcam.shape}')
else:
    print('Baseline Grad-CAM not found — regenerating...')

    def reshape_transform(tensor, h=14, w=14):
        return tensor[:, 1:, :].reshape(tensor.size(0), h, w, -1).permute(0, 3, 1, 2)

    base_ckpt  = torch.load(BASELINE_CKPT, map_location='cpu', weights_only=False)
    base_model = timm.create_model(base_ckpt['hparams']['model'],
                                   pretrained=False, num_classes=14)
    base_model.load_state_dict(base_ckpt['model_state_dict'])
    base_model.eval().to(DEVICE)

    cam = GradCAM(model=base_model, target_layers=[base_model.blocks[-1].norm1],
                  reshape_transform=reshape_transform)

    N_VAL = len(df_valid)
    baseline_gcam = np.zeros((N_VAL, len(LABEL_COLS), 14, 14), dtype=np.float32)

    for img_idx, (imgs, *_) in enumerate(tqdm(val_loader_1, desc='Baseline Grad-CAM')):
        imgs = imgs.to(DEVICE)
        for li in range(len(LABEL_COLS)):
            cam_224 = cam(input_tensor=imgs, targets=[ClassifierOutputTarget(li)])
            baseline_gcam[img_idx, li] = cv2.resize(cam_224[0], (14, 14),
                                                     interpolation=cv2.INTER_AREA)

    np.save(baseline_gcam_path, baseline_gcam)
    print(f'Saved baseline Grad-CAM: {baseline_gcam.shape}')

In [ ]:
# ── Cell 10 — IoU: token masks vs baseline Grad-CAM ──────────────────────────
#
# Core map comparison — the novel contribution.
#
# For each val image × label × keep_ratio:
#   1. Token mask:    [14,14] binary — 1 where patch token survived
#   2. Grad-CAM:      [14,14] float  — class-discriminative heatmap
#   3. Threshold Grad-CAM at top-keep_ratio% to make it binary
#      (so both masks have the same number of 1s by design)
#   4. IoU = intersection / union
#
# Only computed on images that are positive for the label.
# High IoU = model drops uninformative tokens, keeps diagnostic ones.

is_pos = df_valid[LABEL_COLS].values == 1.0   # [202, 14]

def threshold_gcam(cam_14, keep_ratio):
    """Binary mask matching the keeping ratio: top-k% patches = 1."""
    thresh = np.percentile(cam_14, (1 - keep_ratio) * 100)
    return (cam_14 >= thresh).astype(np.float32)

def iou(a, b):
    inter = (a * b).sum()
    union = np.clip(a + b, 0, 1).sum()
    return inter / (union + 1e-8)


iou_rows = []

for keep_ratio in KEEP_RATIOS:
    pct   = int(keep_ratio * 100)
    masks = token_masks[pct]   # [202, 14, 14]

    for li, label in enumerate(LABEL_COLS):
        pos_idx = np.where(is_pos[:, li])[0]
        if len(pos_idx) < RELIABLE_MIN:
            continue

        ious = []
        for img_idx in pos_idx:
            token_mask = masks[img_idx]                          # [14, 14] binary
            cam_map    = baseline_gcam[img_idx, li]             # [14, 14] float
            cam_bin    = threshold_gcam(cam_map, keep_ratio)    # [14, 14] binary
            ious.append(iou(token_mask, cam_bin))

        iou_rows.append({
            'keep_ratio': pct,
            'label':      label,
            'mean_iou':   np.mean(ious),
            'std_iou':    np.std(ious),
            'n_images':   len(ious),
        })

df_iou = pd.DataFrame(iou_rows)
pivot  = df_iou.pivot(index='label', columns='keep_ratio', values='mean_iou')
print('\nMean IoU (token mask vs baseline Grad-CAM) per label:')
print(pivot.round(4).to_string())
print(f'\nOverall mean IoU:')
for pct in [int(r*100) for r in KEEP_RATIOS]:
    m = df_iou[df_iou['keep_ratio'] == pct]['mean_iou'].mean()
    print(f'  {pct}%: {m:.4f}')

df_iou.to_csv(CKPT_DIR / 'token_pruning_iou.csv', index=False)
print('\nSaved token_pruning_iou.csv')

In [ ]:
# ── Cell 11 — Per-subgroup IoU (bias analysis) ────────────────────────────────
#
# Research question 3: does token pruning disproportionately disrupt
# spatial diagnostic reasoning for demographic subgroups?
# Computes mean IoU per (subgroup, label, keep_ratio).

from src.dataset import age_group

df_valid['age_grp'] = df_valid['Age'].apply(age_group)

SUBGROUP_COLS = [
    ('Sex',     ['Male', 'Female']),
    ('age_grp', ['<40', '40-60', '>60']),
]

subgroup_rows = []

for keep_ratio in KEEP_RATIOS:
    pct   = int(keep_ratio * 100)
    masks = token_masks[pct]

    for sg_col, sg_vals in SUBGROUP_COLS:
        for sg_val in sg_vals:
            sg_idx = df_valid.index[df_valid[sg_col] == sg_val].tolist()
            if not sg_idx:
                continue

            for li, label in enumerate(LABEL_COLS):
                pos_sg = [i for i in sg_idx if is_pos[i, li]]
                if len(pos_sg) < 3:
                    continue

                ious = []
                for img_idx in pos_sg:
                    cam_bin = threshold_gcam(baseline_gcam[img_idx, li], keep_ratio)
                    ious.append(iou(masks[img_idx], cam_bin))

                subgroup_rows.append({
                    'keep_ratio': pct,
                    'subgroup':   f'{sg_col}={sg_val}',
                    'label':      label,
                    'mean_iou':   np.mean(ious),
                    'n_images':   len(ious),
                })

df_sg = pd.DataFrame(subgroup_rows)

# Summary: mean IoU per subgroup collapsed over reliable labels
print('Mean IoU per subgroup (collapsed over labels):')
summary = df_sg.groupby(['keep_ratio', 'subgroup'])['mean_iou'].mean().unstack('subgroup')
print(summary.round(4).to_string())

df_sg.to_csv(CKPT_DIR / 'token_pruning_subgroup_iou.csv', index=False)
print('\nSaved token_pruning_subgroup_iou.csv')

In [ ]:
# ── Cell 12 — Summary plots ───────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Left: AUC vs keep ratio
ax = axes[0]
macro_aucs = []
for pct in [int(r*100) for r in KEEP_RATIOS]:
    vals = [auc_table.get(l, {}).get(f'dynvit_{pct}pct', float('nan'))
            for l in LABEL_COLS if int(val_pos.get(l, 0)) >= RELIABLE_MIN]
    with warnings.catch_warnings(): warnings.simplefilter('ignore')
    macro_aucs.append(np.nanmean(vals))
ax.axhline(best_auc, linestyle='--', color='gray', label='Baseline')
ax.plot([int(r*100) for r in KEEP_RATIOS], macro_aucs, 'o-', color='steelblue')
ax.set_xlabel('Keep ratio (%)')
ax.set_ylabel('Macro AUC (reliable labels)')
ax.set_title('AUC vs token keep ratio')
ax.legend(); ax.grid(alpha=0.3)

# Centre: mean IoU per keep ratio
ax = axes[1]
mean_ious = [df_iou[df_iou['keep_ratio'] == int(r*100)]['mean_iou'].mean()
             for r in KEEP_RATIOS]
ax.plot([int(r*100) for r in KEEP_RATIOS], mean_ious, 's-', color='coral')
ax.set_xlabel('Keep ratio (%)')
ax.set_ylabel('Mean IoU (token mask vs Grad-CAM)')
ax.set_title('Spatial alignment: surviving tokens vs Grad-CAM')
ax.set_ylim(0, 1); ax.grid(alpha=0.3)

# Right: IoU per label at 50% keep ratio
ax    = axes[2]
pct50 = df_iou[df_iou['keep_ratio'] == 50].set_index('label')['mean_iou']
pct50.plot(kind='barh', ax=ax, color='teal', alpha=0.8)
ax.set_xlabel('Mean IoU')
ax.set_title('IoU at 50% keep ratio — per label')
ax.set_xlim(0, 1); ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(MAPS_DIR / 'token_pruning_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {MAPS_DIR / "token_pruning_summary.png"}')